# Calibrated checkerboard FRC

Resolution of single SEM images, measured with the **checkerboard one-image FRC** and then put
back onto the **two-image gold-standard scale** using the calibration fitted in
`frc_calibration_checkerboard.ipynb`.

The uncalibrated notebook (`frc_resolution_Argon.ipynb`) reported four measurements per image -
two splits x two windows - and left the choice open. This one does not: it reports the single
recipe the calibration was fitted for, and nothing else.

- **checkerboard split**, both diagonals averaged - the split the calibration corrects;
- **no edge taper** (`FRC_WINDOW = "none"`) - the calibration absorbs the untapered leakage,
  so a window here would measure something the calibration does not describe;
- **two scales**, each with its own fitted curve: the **whole masked region** and
  **256 px patches**;
- **inside the cell mask only**, so what is reported is the resolution on the specimen rather
  than on the resin around it.

Everything is reported twice: `r_co1` is the raw checkerboard measurement, `r_cal` is that value
mapped through `r_ref = a*exp(c*(r_co1 - b)) + d`. `r_cal` is the number to quote; `r_co1` is
kept so the correction applied is always visible.

**The calibration is a lookup, not a re-measurement.** It was fitted on a different specimen
(the calibration test dataset) over a bounded range of `r_co1`, in pixel units. Two things
therefore have to be checked on every row rather than assumed, and both are columns in the
output: whether `r_co1` fell inside the fitted range (`calib_range`), and how close `r_cal` sits
to the curve's own ceiling, the value it saturates at as `r_co1` grows (`ceiling_frac`). A point
at the ceiling is not a measurement of ~6 px; it is the calibration saying "coarser than
anything I was fitted on".

## Configuration

- **`INPUT_PATH`** - folder of input `.tif` images (one image per field of view).
- **`CALIBRATION_DIR`** - the output folder written by `frc_calibration_checkerboard.ipynb`.
  `calibration_fits.csv` is read from it.
- **`OUTPUT_PATH`** - folder for the CSVs and figures (`""` to skip writing).
- **`CELL_MASK_DIR` / `CELL_MASK_TOKEN`** - per-image mask `<raw_stem><token>.tif`. Unlike the
  uncalibrated notebook this is **required**: every measurement here is inside a mask.
- **`PIXEL_SIZE_NM`** - physical pixel size, or `None` to read it per image from the FEI TIFF tag.

The FRC settings below are **not free parameters**. They are the settings the calibration was
fitted with, and the fitted curve is only meaningful for measurements made the same way. They are
spelled out rather than hidden so that any departure is visible, and `Step 1` re-checks them
against the calibration folder.

In [ ]:
INPUT_PATH      = r"../data/images"
CELL_MASK_DIR   = r"../data/masks"
CELL_MASK_TOKEN = "_full_cell"

CALIBRATION_DIR = r"../outputs/checkerboard_calibration"
OUTPUT_PATH     = r"../outputs/calibrated_frc_patch_250"

PIXEL_SIZE_NM = 5.0     # None -> read per image from the FEI_HELIOS TIFF tag

# --- which calibration variant to apply ------------------------------------------------------
# "all" is the fit over every usable calibration point; "unflagged" is the sensitivity refit with
# the near-limit points removed. "all" is the primary result - the flagged points were kept in the
# fit deliberately, because they are the regime where the one-image bias is largest.
CALIBRATION_VARIANT = "all"

# --- the measurement recipe the calibration was fitted for -----------------------------------
FRC_WINDOW  = "none"    # no edge taper. Changing this invalidates the calibration.
THRESHOLD   = "0.143"   # fixed threshold used by both Koho and Dumoux
N_RINGS     = None      # None -> min(image_shape) // 2
SMOOTH_FRAC = 0.02      # root-finding smoothing width, as a fraction of the number of rings
CHECKERBOARD_FLOOR_PX = 4.0    # checkerboard Nyquist period, in original pixels
NYQUIST_FLAG_MULTIPLE = 1.05   # crossings within this multiple of the floor are flagged

# --- geometry -------------------------------------------------------------------------------
PATCH_SIZE = 250        # the patch size actually measured here
# Which fitted curve in calibration_fits.csv is applied to those patches. The calibration
# notebook only ever fitted patch_256, so any other PATCH_SIZE would otherwise find no curve
# and leave every r_cal as NaN. Pinning this to 256 reuses the 256 px fit at other patch
# sizes. That is a borrowed correction, not a re-fit: a 200 px patch has fewer Fourier
# samples per ring than a 256 px one, so its raw r_co1 is biased differently and the 256 px
# curve does not describe that difference. Set this back to PATCH_SIZE once a fit at that
# size exists.
CALIBRATION_PATCH_SIZE = 256
# Fraction of a patch that has to lie inside the mask for it to be measured. 0.0 means "every
# patch that touches the mask at all", i.e. holds at least one masked pixel; 1.0 would keep only
# patches entirely inside the cell. Anything below 1.0 admits patches that are not pure specimen,
# so what happens to the rest of them is a real choice - see PATCH_FILL_OUTSIDE. `mask_coverage`
# is recorded per patch either way, so a stricter cut can be applied to the CSV afterwardss
# without re-measuring anything.
MIN_PATCH_COVERAGE = 0.0
# How a partly covered patch is treated. True replaces its out-of-mask pixels with the patch's
# own in-mask mean, which keeps the measurement inside the cell but hands the FFT a flat,
# perfectly correlated region - and at low coverage that region *is* most of the patch, so the
# FRC ends up describing the shape of the mask rather than the specimen. False measures the patch
# as acquired, which is honest about what was in it but mixes cell with whatever surrounds it.
# Step 5 breaks the outcomes down by coverage so the cost of this choice stays visible.
PATCH_FILL_OUTSIDE = True
MIN_REGION_PX = 64      # skip masked regions whose bounding box is smaller than this

# --- figures ---------------------------------------------------------------------------------
PREVIEW_IDX   = 0       # which image Step 4 draws
SAVE_FIGURES  = True    # write the Step 4 / Step 7 figures into OUTPUT_PATH/figures

# Step 6: one four-panel heatmap per image. Off skips the step entirely, which is worth doing
# while iterating on the measurement, since drawing costs more than measuring does.
SAVE_HEATMAPS         = True
HEATMAPS_SHOWN_INLINE = 2      # how many are drawn in the notebook; the rest are saved only
HEATMAP_DRAW_STEP     = 2      # decimation used for the background image and the mask outline.
                               # Drawing only - it changes nothing that is measured, and 1 gives
                               # full resolution at several times the cost.

# The PNG is a picture: its colour scale is clipped to the 2-98% range and its unmeasured patches
# are drawn in a grey that no longer distinguishes them from a value, so the numbers cannot be
# read back out of it. These write the grids themselves, unclipped, so the maps can be replotted
# or analysed elsewhere. Independent of SAVE_HEATMAPS - the grids come from `patch_df`, so they
# can be written without drawing anything.
SAVE_HEATMAP_DATA = True
HEATMAP_DATA_CSV  = True    # one 2-D grid CSV per image per quantity - opens in anything
HEATMAP_DATA_TIF  = True   # the same grids as float32 TIFF, for ImageJ/napari
# The grids are one cell per patch, so a TIFF written straight from them is PATCH_SIZE times
# smaller than the image and will not overlay it. True writes them on the image's own pixel grid
# instead - same width and height as the raw image, each patch value filled across the pixels it
# covers - so a map can be dropped onto the image it came from directly. False keeps the compact
# form, one TIFF pixel per patch. Either way the pixel size is written into the TIFF tags.
HEATMAP_TIF_FULL_RES = True

print("Input images   :", INPUT_PATH)
print("Cell masks     :", CELL_MASK_DIR, "  token:", CELL_MASK_TOKEN)
print("Calibration    :", CALIBRATION_DIR, " variant:", CALIBRATION_VARIANT)
print("Output         :", OUTPUT_PATH or "(none)")
print("Pixel size     :", f"{PIXEL_SIZE_NM} nm" if PIXEL_SIZE_NM else "(from FEI metadata)")
print("Measurement    : checkerboard split, window = %r, threshold = %s" % (FRC_WINDOW, THRESHOLD))
print("Scales         : whole masked region  +  %d px patches (%s)"
      % (PATCH_SIZE, "every patch touching the mask" if MIN_PATCH_COVERAGE <= 0
         else ">= %.0f%% inside the mask" % (100 * MIN_PATCH_COVERAGE)))
print("Patch curve    : patch_%d fit%s"
      % (CALIBRATION_PATCH_SIZE, "" if CALIBRATION_PATCH_SIZE == PATCH_SIZE
         else "  (borrowed - patches are measured at %d px)" % PATCH_SIZE))
print("Partial patches: %s" % ("out-of-mask pixels filled with the in-mask mean"
                               if PATCH_FILL_OUTSIDE else "measured as acquired, no fill"))
print("Heatmaps       : %s" % ("one per image -> OUTPUT_PATH/heatmaps, %d shown inline"
                               % HEATMAPS_SHOWN_INLINE if SAVE_HEATMAPS else "off"))
print("Heatmap data   : %s" % ("raw grids -> OUTPUT_PATH/heatmaps/data  (npz%s%s)"
                               % (" + csv" if HEATMAP_DATA_CSV else "",
                                  " + tif" if HEATMAP_DATA_TIF else "")
                               if SAVE_HEATMAP_DATA else "off"))
print("Sampling limit : %.1f px (checkerboard Nyquist); flagged below %.2f px"
      % (CHECKERBOARD_FLOOR_PX, NYQUIST_FLAG_MULTIPLE * CHECKERBOARD_FLOOR_PX))

In [ ]:
import os
import time

import numpy as np
import pandas as pd
import tifffile
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda v: f"{v:.3f}")

## Step 1 - load the calibration

`calibration_fits.csv` holds one row per (scale, variant): the four parameters of
`r_ref = a*exp(c*(r_co1 - b)) + d` and the range of `r_co1` the fit was built from. Two scales
were fitted separately - `full_frame` and `patch_256` - because the number of Fourier samples
sets how much the ring binning is smoothed by noise, and a 256 px patch has a sixty-fourth of
the samples of the 2048 px frame it came from. The scale of the measurement therefore has to
match the scale of the curve applied to it, which is why nothing below shares a fit between the
two.

The **ceiling** printed for each fit is `d`, the value the curve saturates at as `r_co1` grows
without limit (the fits have `a < 0` and `c < 0`, so they approach `d` from below). It is the
coarsest resolution this calibration can express. Any measurement mapped to within a few percent
of it is reporting the ceiling, not the specimen.

In [ ]:
FULL_FRAME  = "full_frame"
PATCH_SCALE = f"patch_{CALIBRATION_PATCH_SIZE}"


def calibration_model(r, a, b, c, d):
    """Dumoux/Quoll calibration form: r_ref = a * exp(c * (r - b)) + d, in pixel units."""
    return a * np.exp(c * (np.asarray(r, dtype=np.float64) - b)) + d


fits_path = os.path.join(CALIBRATION_DIR, "calibration_fits.csv")
if not os.path.isfile(fits_path):
    raise FileNotFoundError(
        f"No calibration_fits.csv in {CALIBRATION_DIR}. Run frc_calibration_checkerboard.ipynb "
        f"with OUTPUT_PATH set to that folder first.")

fits_df = pd.read_csv(fits_path)
if (fits_df["units"] != "pixels").any():
    raise ValueError("calibration_fits.csv holds a fit in units other than pixels; the mapping "
                     "below assumes pixel units.")

CALIB = {}
for scale in (FULL_FRAME, PATCH_SCALE):
    sel = fits_df[(fits_df["scale"] == scale) & (fits_df["variant"] == CALIBRATION_VARIANT)]
    if sel.empty:
        print(f"WARNING: no '{CALIBRATION_VARIANT}' fit for scale '{scale}' - measurements at "
              f"that scale will be reported raw, with r_cal left as NaN.")
        continue
    row = sel.iloc[0]
    params = (float(row["a"]), float(row["b"]), float(row["c"]), float(row["d"]))
    # The curve saturates at d when a < 0 and c < 0; otherwise it grows without bound and there
    # is no ceiling to report.
    ceiling = float(row["d"]) if (params[0] < 0 and params[2] < 0) else np.nan
    CALIB[scale] = {"params": params, "x_min": float(row["x_min"]), "x_max": float(row["x_max"]),
                    "resid_std": float(row["resid_std"]), "r2": float(row["r2"]),
                    "n": int(row["n"]), "ceiling_px": ceiling}

if PATCH_SCALE not in CALIB:
    print(f"NOTE: the calibration folder has no fit at {CALIBRATION_PATCH_SIZE} px. Available "
          f"scales: {sorted(fits_df['scale'].unique())}. CALIBRATION_PATCH_SIZE must match "
          f"one of them.")
elif CALIBRATION_PATCH_SIZE != PATCH_SIZE:
    print(f"NOTE: patches are measured at {PATCH_SIZE} px but corrected with the "
          f"{CALIBRATION_PATCH_SIZE} px curve, which was fitted at a different scale.\nThe "
          f"ring binning of a {PATCH_SIZE} px patch is not that of a {CALIBRATION_PATCH_SIZE} "
          f"px one, so r_cal below carries an extra bias\non top of the fit residual that "
          f"nothing here can quantify, and the fitted range is borrowed\ntoo, so calib_range "
          f"is indicative rather than exact.\n")

print(f"Calibration read from {fits_path}\n")
for scale, c in CALIB.items():
    a, b, cc, d = c["params"]
    print(f"{scale}  (variant '{CALIBRATION_VARIANT}', fitted on n = {c['n']} points)")
    print(f"   r_ref = {a:.4f} * exp({cc:.4f} * (r_co1 - {b:.4f})) + {d:.4f}    [pixels]")
    print(f"   fitted over r_co1 = {c['x_min']:.2f} - {c['x_max']:.2f} px"
          f"   ->  r_cal = {calibration_model(c['x_min'], *c['params']):.2f} - "
          f"{calibration_model(c['x_max'], *c['params']):.2f} px")
    print(f"   residual scatter {c['resid_std']:.3f} px   R2 = {c['r2']:.3f}"
          f"   ceiling = {c['ceiling_px']:.2f} px"
          + (f" ({c['ceiling_px'] * PIXEL_SIZE_NM:.1f} nm at {PIXEL_SIZE_NM} nm/px)"
             if PIXEL_SIZE_NM else "") + "\n")

# The recipe the fit was made with has to be the recipe used here. The calibration notebook's
# settings are not stored in the CSV, so this is a reminder rather than a check.
print("The fit above describes a checkerboard measurement made with no window, threshold 0.143,\n"
      "both diagonals averaged, rings binned to Nyquist, and root-finding on a smoothed curve.\n"
      "Step 2 reproduces exactly that. Changing FRC_WINDOW, THRESHOLD or SMOOTH_FRAC would leave\n"
      "the calibration describing a measurement that is no longer the one being made.")

## Step 2 - FRC machinery

Copied unchanged from `frc_calibration_checkerboard.ipynb`. That is deliberate and it is the
whole reason the calibration transfers: the fitted curve maps the output of *this* code onto the
gold standard, so a re-implementation - even a better one - would map something else.

The five choices that differ from a stock FRC are argued in full in the calibration notebook.
In brief: rings stop at Nyquist so the corners of the frequency square cannot host a spuriously
fine crossing; there is no edge taper, because a taper spreads the enormous correlated
low-frequency power into exactly the rings the checkerboard measurement lives in; both
checkerboard diagonals are averaged, which halves the ring-to-ring noise and symmetrises the
one-pixel offset between the two halves; the curve is smoothed before the crossing is found, so
a single noisy ring dipping under 0.143 is not read as the resolution; and a curve that never
crosses returns `NaN` rather than the end of the axis.

In [ ]:
# The innermost rings hold almost no FFT samples - ring 0 holds the DC term alone, which mean
# subtraction has just set to zero, so its "correlation" is 0/0. Ring k holds roughly 2*pi*k
# samples, so this floor discards the first two or three rings only. Not a tuning knob.
MIN_RING_SAMPLES = 16


def apply_window(image, window=None):
    """Edge taper applied before the FFT. Default (`FRC_WINDOW`) is none."""
    window = FRC_WINDOW if window is None else window
    if window in (None, "none"):
        return image
    ny, nx = image.shape
    if window == "hann":
        return image * np.outer(np.hanning(ny), np.hanning(nx))
    if window == "hamming":
        return image * np.outer(np.hamming(ny), np.hamming(nx))
    raise ValueError(f"Unknown window: {window}")


def radial_frequency_map(shape):
    """Radial spatial frequency (cycles / pixel) of every FFT sample, fftfreq convention."""
    fy = np.fft.fftfreq(shape[0])[:, None]
    fx = np.fft.fftfreq(shape[1])[None, :]
    return np.sqrt(fy ** 2 + fx ** 2)


# Ring maps are expensive to build and identical for every image of a given shape, so they are
# cached: this notebook computes thousands of FRC curves on a handful of distinct shapes.
_RING_CACHE = {}


def ring_index(shape, n_rings):
    """Ring binning for one FFT shape -> (keep mask, ring index of kept samples, centres, counts).

    Bin edges run 0 -> 0.5 cyc/px and every sample outside that radius is dropped: those samples
    are the corners of the frequency square, they exist along the diagonals only, and binning
    into them yields 1/f < 2 px - a period finer than the sampling can represent.
    """
    key = (tuple(shape), int(n_rings))
    if key not in _RING_CACHE:
        fmap = radial_frequency_map(shape).ravel()
        edges = np.linspace(0.0, 0.5, n_rings + 1)
        keep = fmap <= 0.5
        idx = np.minimum(np.searchsorted(edges, fmap[keep], side="right") - 1, n_rings - 1)
        centres = 0.5 * (edges[:-1] + edges[1:])
        n_pix = np.bincount(idx, minlength=n_rings).astype(np.float64)
        _RING_CACHE[key] = (keep, idx, centres, n_pix)
    return _RING_CACHE[key]


def fill_undersampled(curve, n_pix):
    """Replace the under-populated innermost rings with the first trustworthy value.

    Left alone they are 0/0 noise of order +/-1, and the smoothing below would spread that over
    the first few percent of the frequency axis.
    """
    curve = np.asarray(curve, dtype=np.float64).copy()
    valid = np.asarray(n_pix) >= MIN_RING_SAMPLES
    if valid.any() and not valid.all():
        curve[:int(np.argmax(valid))] = curve[int(np.argmax(valid))]
    return curve


def frc_curve(image1, image2, n_rings=None, window=None):
    """Raw FRC between two equally-shaped 2-D images.

    Returns (frequencies, frc, n_pixels_per_ring). The frequencies are cycles per sample of the
    arrays passed in; for checkerboard halves that sample is two original pixels wide, which is
    what `px_scale` in `_package` converts back.
    """
    if image1.shape != image2.shape:
        raise ValueError(f"Images must match; got {image1.shape} vs {image2.shape}")

    # The mean is removed so that ring 0 measures structure. In an SEM image the DC level is
    # orders of magnitude above everything else and is trivially correlated between any two
    # halves, which would drag the innermost rings towards 1 for no physical reason.
    img1 = apply_window(np.asarray(image1, dtype=np.float64) - np.mean(image1), window)
    img2 = apply_window(np.asarray(image2, dtype=np.float64) - np.mean(image2), window)

    F1 = np.fft.fft2(img1)
    F2 = np.fft.fft2(img2)

    n_rings = (min(image1.shape) // 2) if n_rings is None else int(n_rings)
    keep, idx, centres, n_pix = ring_index(image1.shape, n_rings)

    num = np.bincount(idx, weights=np.real(F1 * np.conj(F2)).ravel()[keep], minlength=n_rings)
    p1 = np.bincount(idx, weights=(np.abs(F1) ** 2).ravel()[keep], minlength=n_rings)
    p2 = np.bincount(idx, weights=(np.abs(F2) ** 2).ravel()[keep], minlength=n_rings)

    den = np.sqrt(p1 * p2)
    frc_vals = np.where(den > 0, num / np.where(den > 0, den, 1.0), np.nan)
    return centres, fill_undersampled(frc_vals, n_pix), n_pix


def threshold_curve(n_pixels_per_ring, criterion=None):
    """Fixed thresholds ("0.143", "0.5") or the van Heel & Schatz (2005) information curves."""
    criterion = THRESHOLD if criterion is None else criterion
    n = np.maximum(np.asarray(n_pixels_per_ring, dtype=np.float64), 1.0)
    if criterion == "0.143":
        return np.full_like(n, 0.143)
    if criterion == "0.5":
        return np.full_like(n, 0.5)
    if criterion in ("half_bit", "halfbit"):
        snr = 0.2071
    elif criterion in ("one_bit", "onebit"):
        snr = 0.5
    else:
        raise ValueError(f"Unknown threshold criterion: {criterion}")
    sq = np.sqrt(n)
    return (snr + (2.0 * np.sqrt(snr) + 1.0) / sq) / (snr + 1.0 + 2.0 * np.sqrt(snr) / sq)


def smooth_window(n_rings, frac=None):
    """Odd moving-average width used for a curve of `n_rings` points.

    A fixed *fraction* rather than a fixed number of rings keeps the amount of smoothing
    comparable between a whole masked region and a 256 px patch, which matters because the two
    scales are compared against each other below.
    """
    frac = SMOOTH_FRAC if frac is None else frac
    return max(3, int(round(frac * n_rings)) | 1)


def smooth_curve(y, frac=None):
    """Moving average over `smooth_window` rings (reflected edges)."""
    y = np.asarray(y, dtype=np.float64)
    w = smooth_window(len(y), frac)
    if w >= len(y):
        return y.copy()
    padded = np.pad(y, w // 2, mode="reflect")
    return np.convolve(padded, np.ones(w) / w, mode="valid")


def find_crossing(freqs, curve, thresh, n_pix, persist=None):
    """First frequency where *curve* falls below *thresh* and stays below -> (frequency, status).

    `persist` consecutive rings must also be below, so a single dip is not read as the
    resolution, and the numerically degenerate innermost rings cannot host a crossing.

    Two outcomes return NaN rather than a number, because in both there is no crossing to report
    and returning the end of the axis would invent one:

    - "beyond_floor" - the curve never crosses, so the resolution is finer than the finest period
      these samples can carry. Censored from below, not equal to the floor.
    - "no_signal"    - already below the threshold at the first eligible ring.
    """
    freqs = np.asarray(freqs, dtype=np.float64)
    thresh = np.asarray(thresh, dtype=np.float64)
    n = freqs.size
    persist = max(3, n // 50) if persist is None else int(persist)

    usable = np.asarray(n_pix) >= MIN_RING_SAMPLES
    if not usable.any():
        return np.nan, "no_signal"
    i_min = int(np.argmax(usable))

    below = np.asarray(curve, dtype=np.float64) < thresh
    # Rolling "all below" over the next `persist` rings, via a cumulative count of the not-belows.
    csum = np.concatenate(([0], np.cumsum(~below)))
    stays = (csum[np.minimum(np.arange(n) + persist, n)] - csum[:n]) == 0
    stays[:i_min] = False

    if not stays.any():
        return np.nan, "beyond_floor"
    i = int(np.argmax(stays))
    if i == i_min:
        return np.nan, "no_signal"

    d_lo = curve[i - 1] - thresh[i - 1]
    d_hi = curve[i] - thresh[i]
    denom = d_lo - d_hi
    if abs(denom) < 1e-12:
        return float(freqs[i]), "ok"
    return float(freqs[i - 1] + (freqs[i] - freqs[i - 1]) * d_lo / denom), "ok"


print("FRC core ready: rings binned to Nyquist only, window = %r, root-finding on a curve "
      "smoothed over %.1f%% of the rings." % (FRC_WINDOW, 100 * SMOOTH_FRAC))

In [ ]:
def split_checkerboard(image, diagonal=0):
    """One of the two checkerboard sub-lattices of *image* (Koho et al. 2019).

    `diagonal=0` pairs the (even, even) pixels with the (odd, odd) ones; `diagonal=1` pairs
    (even, odd) with (odd, even). Either way each half is a square lattice of twice the original
    spacing, so its Nyquist period is 4 original pixels - the floor this whole method sits on.
    """
    img = np.asarray(image, dtype=np.float64)
    if diagonal == 0:
        a, b = img[0::2, 0::2], img[1::2, 1::2]
    elif diagonal == 1:
        a, b = img[0::2, 1::2], img[1::2, 0::2]
    else:
        raise ValueError("diagonal must be 0 or 1")
    h = min(a.shape[0], b.shape[0])
    w = min(a.shape[1], b.shape[1])
    return a[:h, :w], b[:h, :w]


def _package(freqs, raw, n_pix, threshold, pixel_size_nm, split, px_scale, floor_px):
    """Turn a curve into a result dict: smoothing, threshold, crossing, resolution, flagging."""
    smoothed = smooth_curve(raw)
    thr = threshold_curve(n_pix, threshold)
    f_cross, status = find_crossing(freqs, smoothed, thr, n_pix)

    # `freqs` counts cycles per sample of whatever was correlated; px_scale (2 for the
    # checkerboard halves) puts it back on the original pixel grid.
    res_px = px_scale / f_cross if np.isfinite(f_cross) and f_cross > 0 else np.nan
    res_nm = res_px * pixel_size_nm if pixel_size_nm else np.nan

    # A crossing this close to the limit was found in the last few rings, where the smoothing
    # window reflects and the persistence rule runs out of axis.
    near_limit = bool(np.isfinite(res_px) and res_px < NYQUIST_FLAG_MULTIPLE * floor_px)

    unit = "nm" if pixel_size_nm else "px"
    scale = pixel_size_nm if pixel_size_nm else 1.0
    if status == "beyond_floor":
        res_str = f"<= {floor_px * scale:.2f} {unit} (never crosses; at the sampling limit)"
    elif status == "no_signal":
        res_str = "no correlated signal"
    elif near_limit:
        res_str = f"{res_px * scale:.2f} {unit} (flagged: within {NYQUIST_FLAG_MULTIPLE:g}x the limit)"
    else:
        res_str = f"{res_px * scale:.2f} {unit}"

    return {
        "split": split,
        "frequencies": freqs,                    # cycles per sample of the correlated arrays
        "freq_cyc_per_px": freqs / px_scale,     # cycles per pixel of the original image
        "frc_raw": raw,
        "frc_smooth": smoothed,
        "threshold_curve": thr,
        "threshold_name": threshold,
        "n_pixels_per_ring": n_pix,
        "crossing_freq": f_cross,
        "crossing_freq_orig": f_cross / px_scale if np.isfinite(f_cross) else np.nan,
        "status": status,
        "near_limit": near_limit,
        "resolution_px": res_px,
        "resolution_nm": res_nm,
        "resolution_str": res_str,
        "pixel_size_nm": pixel_size_nm,
        "px_scale": px_scale,
        "floor_px": floor_px,
    }


def frc_checkerboard(image, threshold=None, n_rings=None, pixel_size_nm=None, window=None,
                     average_diagonals=True):
    """One-image FRC by checkerboard split, both diagonals averaged (Koho 2019 / Dumoux 2023).

    The two diagonal splits are independent estimates of the same curve, so averaging them halves
    the ring-to-ring noise as well as symmetrising the half-pixel offset between the halves.
    """
    curves = []
    for diagonal in ((0, 1) if average_diagonals else (0,)):
        a, b = split_checkerboard(image, diagonal)
        freqs, raw, n_pix = frc_curve(a, b, n_rings=n_rings, window=window)
        curves.append(raw)
    return _package(freqs, np.mean(curves, axis=0), n_pix, threshold or THRESHOLD, pixel_size_nm,
                    split="checkerboard", px_scale=2.0, floor_px=CHECKERBOARD_FLOOR_PX)


def calibrate(r_co1_px, scale):
    """Map a raw checkerboard resolution onto the gold-standard scale -> (r_cal_px, range, ceiling_frac).

    `range` records where `r_co1` sat relative to the fitted interval, because that - not the
    fit statistics - is what decides whether the returned number means anything:

    - "in_range"           - interpolation, the case the fit describes;
    - "extrapolated_high"  - coarser than anything fitted. The curve saturates at its ceiling, so
      the answer stops responding to the measurement: it reads "coarser than the fit can say";
    - "extrapolated_low"   - finer than anything fitted, where the exponential turns over steeply;
    - "no_fit" / "no_value" - no curve for this scale, or nothing to correct.

    `ceiling_frac` is r_cal as a fraction of the curve's asymptote. Near 1 the value is the
    ceiling rather than the specimen.
    """
    cal = CALIB.get(scale)
    if cal is None:
        return np.nan, "no_fit", np.nan
    if not np.isfinite(r_co1_px):
        return np.nan, "no_value", np.nan

    val = float(calibration_model(r_co1_px, *cal["params"]))
    if r_co1_px < cal["x_min"]:
        rng = "extrapolated_low"
    elif r_co1_px > cal["x_max"]:
        rng = "extrapolated_high"
    else:
        rng = "in_range"
    if not np.isfinite(val) or val <= 0:
        return np.nan, rng, np.nan
    frac = val / cal["ceiling_px"] if np.isfinite(cal["ceiling_px"]) and cal["ceiling_px"] > 0 else np.nan
    return val, rng, frac


def plot_frc(result, title="", ax=None, show_raw=True, color="steelblue", label=None):
    """FRC against frequency in cycles per *original* pixel."""
    if ax is None:
        _, ax = plt.subplots(figsize=(7, 4.5))
    f = result["freq_cyc_per_px"]
    if show_raw:
        ax.plot(f, result["frc_raw"], lw=0.7, alpha=0.3, color=color)
    ax.plot(f, result["frc_smooth"], lw=1.7, color=color, label=label or "FRC (checkerboard)")
    ax.plot(f, result["threshold_curve"], "--", color="crimson", lw=1.1,
            label=f"threshold ({result['threshold_name']})")
    ax.axhline(0.0, color="grey", lw=0.8)

    floor_f = 1.0 / result["floor_px"]
    ax.axvspan(floor_f, 0.5, color="grey", alpha=0.12)
    ax.axvline(floor_f, color="grey", lw=1.0, ls="-.",
               label=f"sampling floor ({result['floor_px']:.0f} px)")
    if np.isfinite(result["crossing_freq_orig"]):
        ax.axvline(result["crossing_freq_orig"], color="green", ls=":", lw=1.5,
                   label=f"$r_{{co1}}$ = {result['resolution_str']}")

    ax.set_xlabel("Spatial frequency (cycles / original pixel)")
    ax.set_ylabel("FRC")
    ax.set_xlim(0, 0.5)
    ax.set_ylim(-0.2, 1.05)
    ax.set_title(title, fontsize=10)
    ax.legend(loc="upper right", fontsize=8)
    return ax


print("Measurement API ready: frc_checkerboard, calibrate, plot_frc")

## Step 3 - images, masks, and the patches inside them

Same mask convention as the rest of these notebooks: for a raw `<stem>.tif` the mask is
`<stem><CELL_MASK_TOKEN>.tif`, found anywhere under `CELL_MASK_DIR`.

Two ways of measuring "inside the mask", one per scale, and they are not equivalent:

**Whole masked region.** The FFT needs a rectangle, so the image is cropped to the mask's
bounding box and any pixel inside the box but outside the mask is replaced by the in-mask mean.
That keeps a hard mask edge from injecting a bright cross of high-frequency power, but it does
put a flat, perfectly correlated region into the FFT wherever the cell does not fill its own
bounding box. The `mask_coverage` column records how much of the box that was, and it is the
number to look at before trusting this scale. It is reported because it is the direct analogue
of the previous notebook's in-mask number.

**256 px patches.** The patch grid is laid over the whole image and every patch that touches the
mask is measured - `MIN_PATCH_COVERAGE = 0.0` means one masked pixel is enough. That gives
complete coverage of the cell, including the patches straddling its boundary, which is what the
Step 6 heatmaps need. The price is that a boundary patch is only partly specimen: with
`PATCH_FILL_OUTSIDE` its non-cell pixels are replaced by the in-mask mean, so at low coverage the
FFT is looking mostly at a flat region shaped like the mask rather than at the cell. Step 5
tabulates the outcome against coverage so the size of that effect is measured rather than
guessed, and `mask_coverage` is in the CSV so a stricter cut can be applied afterwards. Raising
`MIN_PATCH_COVERAGE` back to 1.0 keeps only whole-cell patches, which is the geometry the patch
calibration was fitted on.

In [ ]:
SKIP_TOKENS = ("_full_cell", "_Probabilities", "_Simple Segmentation", "_mask")


def find_tif_files(root):
    """All .tif/.tiff files under root, excluding obvious mask / probability companions."""
    paths = []
    for dirpath, _, filenames in os.walk(root):
        for fname in filenames:
            if not fname.lower().endswith((".tif", ".tiff")):
                continue
            if any(tok in fname for tok in SKIP_TOKENS):
                continue
            paths.append(os.path.join(dirpath, fname))
    return sorted(paths)


def load_image_2d(path):
    """Load a TIFF as a 2-D float array (middle slice of a stack; first channel of RGB)."""
    img = tifffile.imread(path)
    if img.ndim == 2:
        return img.astype(np.float64)
    if img.ndim == 3:
        return (img[..., 0] if img.shape[-1] <= 4 else img[img.shape[0] // 2]).astype(np.float64)
    raise ValueError(f"Unsupported image shape {img.shape} for {path}")


def read_pixel_size_nm(path):
    """Pixel size in nm from the FEI_HELIOS TIFF tag (NaN if absent)."""
    try:
        with tifffile.TiffFile(path) as tf:
            tag = tf.pages[0].tags.get("FEI_HELIOS")
            if tag is None:
                return np.nan
            return float(tag.value.get("Scan", {}).get("PixelWidth")) * 1e9
    except (TypeError, ValueError, KeyError):
        return np.nan


def pixel_size_for(path):
    """Configured pixel size, or the one embedded in the file when PIXEL_SIZE_NM is None."""
    if PIXEL_SIZE_NM is not None:
        return float(PIXEL_SIZE_NM)
    px = read_pixel_size_nm(path)
    if not np.isfinite(px):
        raise ValueError(f"No pixel size in the metadata of {path}; set PIXEL_SIZE_NM explicitly.")
    return px


def find_cell_mask(raw_path, cell_mask_dir=None, token=None):
    """Locate <raw_stem><token>.tif under cell_mask_dir (recursive). None if absent."""
    cell_mask_dir = CELL_MASK_DIR if cell_mask_dir is None else cell_mask_dir
    token = CELL_MASK_TOKEN if token is None else token
    if not cell_mask_dir:
        return None
    stem = os.path.splitext(os.path.basename(raw_path))[0]
    target = stem + token + ".tif"
    for dirpath, _, filenames in os.walk(cell_mask_dir):
        if target in filenames:
            return os.path.join(dirpath, target)
    return None


def load_cell_mask(cell_mask_path):
    """Load a cell mask as a boolean array (any non-zero pixel = inside the cell)."""
    cm = tifffile.imread(cell_mask_path)
    if cm.ndim > 2:
        cm = cm[0] if cm.shape[0] < cm.shape[-1] else cm[..., 0]
    return cm > 0


def crop_to_mask(image, mask, fill="mean"):
    """Tight bounding-box crop around the mask, out-of-mask pixels filled with the in-mask mean.

    Returns (crop, mask_crop). The fill is what stops the mask edge from behaving like a step
    function in the FFT; `mask_coverage` downstream records how much of the crop it replaced.
    """
    m = np.asarray(mask) > 0
    rows = np.where(m.any(axis=1))[0]
    cols = np.where(m.any(axis=0))[0]
    if rows.size == 0 or cols.size == 0:
        return None, None
    crop = image[rows[0]:rows[-1] + 1, cols[0]:cols[-1] + 1].astype(np.float64).copy()
    mcrop = m[rows[0]:rows[-1] + 1, cols[0]:cols[-1] + 1]
    if fill == "mean" and (~mcrop).any() and mcrop.any():
        crop[~mcrop] = crop[mcrop].mean()
    return crop, mcrop


def mask_patches(image, mask, size=None, min_coverage=None):
    """Patches of the grid over *image* that touch *mask*.

    Yields (row, col, y0, x0, coverage, patch). Only whole patches that fit inside the image are
    considered, so a partial patch at the right or bottom edge of the frame is never measured -
    the calibration is tied to a 256 px measurement and a smaller one is a different quantity.

    A patch holding no masked pixel at all is never yielded whatever `min_coverage` says: "inside
    the cell" has to mean at least one pixel of cell. Beyond that the coverage rule is
    `MIN_PATCH_COVERAGE`, and `PATCH_FILL_OUTSIDE` decides what happens to the part of a patch
    that is not cell. Coverage is yielded with every patch because at the permissive end of that
    setting it is the number that says how much of the measurement is specimen.
    """
    size = PATCH_SIZE if size is None else size
    min_coverage = MIN_PATCH_COVERAGE if min_coverage is None else min_coverage
    m = np.asarray(mask) > 0
    ny, nx = image.shape[0] // size, image.shape[1] // size
    for i in range(ny):
        for j in range(nx):
            y0, x0 = i * size, j * size
            pm = m[y0:y0 + size, x0:x0 + size]
            if not pm.any():
                continue
            cov = float(pm.mean())
            if cov < min_coverage:
                continue
            patch = image[y0:y0 + size, x0:x0 + size].astype(np.float64).copy()
            if PATCH_FILL_OUTSIDE and cov < 1.0:
                patch[~pm] = patch[pm].mean()
            yield i, j, y0, x0, cov, patch


TIF_PATHS = find_tif_files(INPUT_PATH) if INPUT_PATH else []
print(f"Found {len(TIF_PATHS)} image(s) under {INPUT_PATH}")

MASKED = [p for p in TIF_PATHS if find_cell_mask(p)]
print(f"{len(MASKED)} of them have a matching '{CELL_MASK_TOKEN}' mask under {CELL_MASK_DIR}")
if not MASKED:
    raise RuntimeError("No image has a cell mask; every measurement in this notebook is in-mask.")
missing = [os.path.basename(p) for p in TIF_PATHS if p not in MASKED]
if missing:
    print(f"No mask, so skipped: {', '.join(missing[:8])}"
          + (f" ... (+{len(missing) - 8} more)" if len(missing) > 8 else ""))

## Step 4 - one image, end to end

The machinery on a single image before it is run over the whole folder: what the mask covers,
which patches survive `MIN_PATCH_COVERAGE`, the FRC curve of the masked region, and the
patch-by-patch calibrated resolution as a map. `PREVIEW_IDX` selects the image.

The map is worth more than its median. Resolution varies across a field - focus, charging,
thickness - and a single number per image hides that. If the map is flat, the median is a fair
summary; if one corner is systematically coarser, the median is a compromise between two
regimes rather than a measurement of either.

In [ ]:
prev_path = MASKED[PREVIEW_IDX % len(MASKED)]
prev_px = pixel_size_for(prev_path)
prev_img = load_image_2d(prev_path)
prev_mask = load_cell_mask(find_cell_mask(prev_path))
if prev_mask.shape != prev_img.shape:
    raise ValueError(f"Mask shape {prev_mask.shape} != image shape {prev_img.shape}")

crop, mcrop = crop_to_mask(prev_img, prev_mask)
whole = frc_checkerboard(crop, n_rings=N_RINGS, pixel_size_nm=prev_px)
w_cal, w_rng, w_frac = calibrate(whole["resolution_px"], FULL_FRAME)

ny, nx = prev_img.shape[0] // PATCH_SIZE, prev_img.shape[1] // PATCH_SIZE
pmap = np.full((ny, nx), np.nan)
prev_patches = []
for i, j, y0, x0, cov, patch in mask_patches(prev_img, prev_mask):
    r = frc_checkerboard(patch, n_rings=N_RINGS, pixel_size_nm=prev_px)
    cal_px, rng, frac = calibrate(r["resolution_px"], PATCH_SCALE)
    pmap[i, j] = cal_px * prev_px if np.isfinite(cal_px) else np.nan
    prev_patches.append({"i": i, "j": j, "y0": y0, "x0": x0, "cov": cov, "result": r,
                         "r_cal_nm": pmap[i, j], "range": rng})

print(f"{os.path.relpath(prev_path, INPUT_PATH)}   ({prev_px:.1f} nm/px, image {prev_img.shape})")
print(f"  mask: {100 * prev_mask.mean():5.1f}% of the frame | bounding box {crop.shape} with "
      f"{100 * mcrop.mean():.1f}% of the box inside the mask (the rest is filled)")
print(f"  whole masked region : r_co1 = {whole['resolution_str']:<48s} [{whole['status']}]")
print(f"                        r_cal = "
      + (f"{w_cal * prev_px:.2f} nm ({w_cal:.2f} px)   [{w_rng}"
         + (f", {100 * w_frac:.0f}% of the ceiling]" if np.isfinite(w_frac) else "]")
         if np.isfinite(w_cal) else f"-   [{w_rng}]"))
kept = len(prev_patches)
vals = np.array([p["r_cal_nm"] for p in prev_patches], dtype=np.float64)
finite = vals[np.isfinite(vals)]
print(f"  patches             : {kept} of the {ny * nx} whole patches in the frame touch the mask "
      f"(coverage >= {MIN_PATCH_COVERAGE:.2f}); {finite.size} produced a calibrated number")
if finite.size:
    print(f"                        r_cal median {np.median(finite):.2f} nm, "
          f"range {finite.min():.2f} - {finite.max():.2f} nm")

fig, axes = plt.subplots(2, 2, figsize=(13, 10.5))

lo, hi = np.percentile(prev_img, [1, 99])
axes[0, 0].imshow(prev_img, cmap="gray", vmin=lo, vmax=hi)
axes[0, 0].contour(prev_mask.astype(float), levels=[0.5], colors="yellow", linewidths=1.2)
for p in prev_patches:
    axes[0, 0].add_patch(Rectangle((p["x0"], p["y0"]), PATCH_SIZE, PATCH_SIZE, fill=False,
                                   edgecolor="deepskyblue", linewidth=0.9))
axes[0, 0].set_title(f"mask (yellow) and the {kept} measured patches (blue)", fontsize=10)
axes[0, 0].axis("off")

if finite.size:
    im = axes[0, 1].imshow(pmap, cmap="viridis_r", interpolation="nearest")
    plt.colorbar(im, ax=axes[0, 1], fraction=0.046, label="$r_{cal}$ (nm)")
axes[0, 1].set_title(f"calibrated resolution per {PATCH_SIZE} px patch", fontsize=10)
axes[0, 1].set_xticks([]); axes[0, 1].set_yticks([])

plot_frc(whole, title=f"whole masked region ({crop.shape[0]} x {crop.shape[1]} px)",
         ax=axes[1, 0], color="darkorange")

if prev_patches:
    order = np.argsort([p["r_cal_nm"] if np.isfinite(p["r_cal_nm"]) else np.inf
                        for p in prev_patches])
    mid = prev_patches[order[len(order) // 2]]
    plot_frc(mid["result"], title=f"median patch (row {mid['i']}, col {mid['j']})",
             ax=axes[1, 1], color="steelblue")

fig.suptitle(f"{os.path.relpath(prev_path, INPUT_PATH)}  -  {prev_px:.1f} nm/px", fontsize=12)
fig.tight_layout()
if OUTPUT_PATH and SAVE_FIGURES:
    os.makedirs(os.path.join(OUTPUT_PATH, "figures"), exist_ok=True)
    fig.savefig(os.path.join(OUTPUT_PATH, "figures", "preview_image.png"), dpi=150)
plt.show()

## Step 5 - batch over the folder

Every image with a mask, at both scales. One row per image for the masked region, one row per
patch for the patch grid, and nothing is filtered on the way out: rows whose FRC never crossed
the threshold keep their `status`, and rows whose `r_co1` fell outside the fitted range keep
their `calib_range`. A table that has already dropped those cannot be audited afterwards, and
the count of them is itself a result - it is the record of how much of this dataset the method
could not measure.

In [ ]:
def measure_image(path):
    """Both scales for one image -> (whole-region row, list of patch rows)."""
    px_nm = pixel_size_for(path)
    image = load_image_2d(path)
    mask = load_cell_mask(find_cell_mask(path))
    meta = {"file": os.path.basename(path),
            "rel_dir": os.path.relpath(os.path.dirname(path), INPUT_PATH),
            "path": path, "px_nm": px_nm}
    if mask.shape != image.shape:
        return {**meta, "status": "mask_shape_mismatch"}, []

    def row_for(result, scale, extra):
        cal_px, rng, frac = calibrate(result["resolution_px"], scale)
        return {**meta, **extra, "scale": scale,
                "r_co1_px": result["resolution_px"], "r_co1_nm": result["resolution_nm"],
                "r_cal_px": cal_px, "r_cal_nm": cal_px * px_nm if np.isfinite(cal_px) else np.nan,
                "status": result["status"], "near_limit": result["near_limit"],
                "calib_range": rng, "ceiling_frac": frac}

    crop, mcrop = crop_to_mask(image, mask)
    if crop is None or min(crop.shape) < MIN_REGION_PX:
        whole_row = {**meta, "scale": FULL_FRAME, "status": "region_too_small"}
    else:
        r = frc_checkerboard(crop, n_rings=N_RINGS, pixel_size_nm=px_nm)
        whole_row = row_for(r, FULL_FRAME,
                            {"crop_h": crop.shape[0], "crop_w": crop.shape[1],
                             "mask_coverage": float(mcrop.mean()),
                             "mask_frac_of_frame": float(mask.mean())})

    patch_rows = []
    for i, j, y0, x0, cov, patch in mask_patches(image, mask):
        r = frc_checkerboard(patch, n_rings=N_RINGS, pixel_size_nm=px_nm)
        patch_rows.append(row_for(r, PATCH_SCALE,
                                  {"row": i, "col": j, "y0": y0, "x0": x0,
                                   "mask_coverage": cov}))
    return whole_row, patch_rows


whole_rows, patch_rows = [], []
t0 = time.time()
for n, path in enumerate(MASKED, 1):
    try:
        wr, prs = measure_image(path)
    except Exception as exc:
        print(f"  FAILED {path}: {exc}")
        continue
    whole_rows.append(wr)
    patch_rows.extend(prs)

    cal = wr.get("r_cal_nm", np.nan)
    got = [p["r_cal_nm"] for p in prs if np.isfinite(p.get("r_cal_nm", np.nan))]
    print(f"[{n:3d}/{len(MASKED)}] {os.path.join(wr['rel_dir'], wr['file'])}")
    print(f"          region : r_co1 = {wr.get('r_co1_nm', np.nan):6.2f} nm  ->  r_cal = "
          f"{cal:6.2f} nm   [{wr.get('status')}, {wr.get('calib_range')}]"
          if np.isfinite(wr.get("r_co1_nm", np.nan)) else
          f"          region : no number   [{wr.get('status')}]")
    if got:
        print(f"          patches: {len(got):3d} of {len(prs):3d} measured   r_cal median "
              f"{np.median(got):6.2f} nm   IQR {np.percentile(got, 75) - np.percentile(got, 25):.2f} nm")
    else:
        print(f"          patches: none of {len(prs)} produced a number")

whole_df = pd.DataFrame(whole_rows)
patch_df = pd.DataFrame(patch_rows)
print(f"\n{len(whole_df)} masked regions and {len(patch_df)} patches in {time.time() - t0:.0f} s")

for name, df in (("whole masked region", whole_df), (f"{PATCH_SIZE} px patches", patch_df)):
    print(f"\n{name}:")
    if df.empty:
        print("  nothing measured.")
        continue
    ok = df[df["status"] == "ok"]
    print(f"  {len(ok)} of {len(df)} crossed the threshold"
          f"   [{int((df['status'] == 'beyond_floor').sum())} never cross, "
          f"{int((df['status'] == 'no_signal').sum())} no signal, "
          f"{int(df['near_limit'].fillna(False).sum())} flagged near the limit]")
    if len(ok):
        print(f"  r_co1 : median {ok['r_co1_nm'].median():6.2f} nm "
              f"({ok['r_co1_px'].median():.2f} px)")
        print(f"  r_cal : median {ok['r_cal_nm'].median():6.2f} nm "
              f"({ok['r_cal_px'].median():.2f} px)")
        for rng, g in ok.groupby("calib_range"):
            print(f"      {rng:18s} {len(g):5d} rows   r_cal median {g['r_cal_nm'].median():6.2f} nm"
                  + (f"   at {100 * g['ceiling_frac'].median():.0f}% of the ceiling"
                     if g["ceiling_frac"].notna().any() else ""))

# Every patch touching the mask is measured, so the grid now includes boundary patches that are
# only partly cell. Whether that is harmless or fatal depends on how much of the patch was filled
# in, which is a thing to measure rather than assume: a coverage band whose measurements mostly
# fail, or whose median drifts away from the full-coverage band, is the fill talking.
if not patch_df.empty:
    edges = [0.0, 0.05, 0.25, 0.50, 0.75, 0.999, 1.0]
    names = ["<5%", "5-25%", "25-50%", "50-75%", "75-99%", "all cell"]
    cov_bin = pd.cut(patch_df["mask_coverage"], bins=edges, labels=names, include_lowest=True)
    print("\nPatches by how much of them is cell:")
    for lbl, g in patch_df.groupby(cov_bin, observed=True):
        ok = g[g["status"] == "ok"]
        med = ok["r_co1_nm"].median() if len(ok) else np.nan
        print(f"  {str(lbl):>8s}  {len(g):5d} patches   {len(ok):5d} measured   "
              f"r_co1 median {med:6.2f} nm   "
              f"[{int((g['status'] == 'beyond_floor').sum())} never cross, "
              f"{int((g['status'] == 'no_signal').sum())} no signal]")

hi = patch_df[patch_df.get("calib_range", "") == "extrapolated_high"] if not patch_df.empty else patch_df
if len(hi):
    print(f"\n{len(hi)} patch(es) measured coarser than anything the calibration was fitted on. "
          f"For those\nthe curve has saturated, so r_cal reads its ceiling and means 'coarser "
          f"than the fit can say'\nrather than a resolution. They are kept and flagged rather "
          f"than dropped - Step 6 reports the\nmedian both with and without them, and the "
          f"in-range median is the one that leaves them out.")

## Step 6 - a heatmap per image

One figure per image, written to `OUTPUT_PATH/heatmaps`, built from `patch_df` - so this step can
be re-run on its own to change the drawing without repeating any FRC.

Four panels, and the pairing is the point:

1. the image with the mask outline and the patches that were measured;
2. **mask coverage** per patch - how much of each patch was actually cell;
3. **`r_co1`**, the raw checkerboard measurement;
4. **`r_cal`**, after calibration.

Read 2 next to 3 and 4. A boundary patch that reads oddly and also reads low on coverage is
reporting the fill, not the specimen. Read 3 next to 4 and the saturation is visible directly:
where the calibration has run out of range, panel 4 goes flat while panel 3 still has structure.
Each panel is scaled to its own data, so absolute colours are not comparable between the two
resolution maps - the spatial pattern within each is what these are for.

**The grids are written out as numbers too**, into `OUTPUT_PATH/heatmaps/data`, one set per image
and sharing the filename stem of the PNG beside it. The picture is not the data: its colour scale
is clipped to the 2nd-98th percentile so that one bad boundary patch cannot flatten the pattern,
and unmeasured patches are drawn in a grey that no longer distinguishes them from a value. Neither
is reversible. What is saved is the unclipped grid, `NaN` wherever there is no number, indexed so
that cell `(i, j)` is image pixels `[i*PATCH_SIZE : (i+1)*PATCH_SIZE, j*PATCH_SIZE :
(j+1)*PATCH_SIZE]` with row 0 at the top of the frame.

| file | holds |
|---|---|
| `<stem>.npz` | every grid at once, plus pixel size, patch size, image shape and the plotting extent |
| `<stem>__<quantity>.csv` | one 2-D grid per file, row/column labelled, blank where unmeasured |
| `<stem>__<quantity>.tif` | the same grids as float32, on the image's own pixel grid so they overlay the raw image (`HEATMAP_TIF_FULL_RES=False` for one pixel per patch instead) |

Six numeric quantities are saved, not the three that are drawn: `mask_coverage`, `r_co1_px`,
`r_co1_nm`, `r_cal_px`, `r_cal_nm` and `ceiling_frac`. The two string grids `status` and
`calib_range` go with them, because a `NaN` on its own cannot say whether a patch was never
measured or was measured and never crossed the threshold - and given how much of this dataset sits
past the end of the calibration, `calib_range` is the grid that says which parts of an `r_cal` map
are interpolation and which are the ceiling.

`SAVE_HEATMAP_DATA` is independent of `SAVE_HEATMAPS`: the grids come from `patch_df`, so they can
be written without drawing anything, which is much the faster way to re-export after a re-run.


In [ ]:
HEATMAP_DIR      = os.path.join(OUTPUT_PATH, "heatmaps") if OUTPUT_PATH else ""
HEATMAP_DATA_DIR = os.path.join(HEATMAP_DIR, "data") if HEATMAP_DIR else ""

# The numeric per-patch columns worth having as a grid. `mask_coverage`, `r_co1_nm` and `r_cal_nm`
# are the three panels drawn below; the others ride along so a saved grid answers the same
# questions the CSV does without having to join back to it.
GRID_KEYS = ("mask_coverage", "r_co1_px", "r_co1_nm", "r_cal_px", "r_cal_nm", "ceiling_frac")
GRID_TEXT_KEYS = ("status", "calib_range")


def patch_maps(df, image_shape, keys=GRID_KEYS):
    """Per-patch columns put back on the patch grid -> 2-D arrays, NaN where nothing was measured."""
    ny, nx = image_shape[0] // PATCH_SIZE, image_shape[1] // PATCH_SIZE
    rows = df["row"].to_numpy(dtype=int)
    cols = df["col"].to_numpy(dtype=int)
    maps = {}
    for k in keys:
        if k not in df.columns:
            continue
        m = np.full((ny, nx), np.nan)
        m[rows, cols] = pd.to_numeric(df[k], errors="coerce").to_numpy(dtype=float)
        maps[k] = m
    return maps


def patch_text_maps(df, image_shape, keys=GRID_TEXT_KEYS):
    """The string columns on the same grid. An empty string marks a patch that was never measured.

    `status` and `calib_range` are the difference between a patch that has no number because the
    FRC never crossed the threshold and one that has no number because it was outside the mask, so
    they travel with the grids rather than being left behind in the CSV.
    """
    ny, nx = image_shape[0] // PATCH_SIZE, image_shape[1] // PATCH_SIZE
    rows = df["row"].to_numpy(dtype=int)
    cols = df["col"].to_numpy(dtype=int)
    maps = {}
    for k in keys:
        if k not in df.columns:
            continue
        m = np.full((ny, nx), "", dtype=object)
        m[rows, cols] = df[k].astype(str).to_numpy()
        maps[k] = m.astype(str)
    return maps


def image_stem(df):
    """Filename stem shared by an image's PNG and its grids, so the two always pair up on disk."""
    rel = os.path.join(df["rel_dir"].iloc[0], df["file"].iloc[0])
    return rel.replace(os.sep, "_").replace(" ", "").replace(".tif", "")


def grid_to_image_raster(m, image_shape):
    """A per-patch grid blown up onto the image's own pixel grid -> float32 array.

    The grid itself is one cell per patch, so a TIFF written straight from it is `PATCH_SIZE`
    times smaller than the image it describes and will not overlay it. Here every pixel of patch
    (i, j) carries that patch's value, so the result is exactly `image_shape` and drops onto the
    raw image in ImageJ or napari without any rescaling. Nothing is interpolated - the value is
    constant across the patch it was measured from, which is all that was measured.

    The strip along the right and bottom edges that no whole patch covered stays NaN, as do
    patches that produced no number.
    """
    full = np.full(image_shape[:2], np.nan, dtype=np.float32)
    block = np.repeat(np.repeat(np.asarray(m, dtype=np.float32), PATCH_SIZE, axis=0),
                      PATCH_SIZE, axis=1)
    h = min(full.shape[0], block.shape[0])
    w = min(full.shape[1], block.shape[1])
    full[:h, :w] = block[:h, :w]
    return full


def save_patch_grids(df, maps, text_maps, image_shape):
    """Write the numbers behind the heatmap, unclipped, one grid cell per patch -> [paths].

    Three forms, because they get used differently:

    - `<stem>.npz` - every grid plus the geometry and the pixel size. `np.load()` returns a
      dict-like of 2-D arrays, so a map comes back in one line with nothing to reconstruct.
    - `<stem>__<quantity>.csv` - one 2-D grid per file, row and column indices labelled, empty
      cell where the patch produced no number. Opens in Excel, Origin, R, anything.
    - `<stem>__<quantity>.tif` - the same grids as float32, on the image's own pixel grid
      (`HEATMAP_TIF_FULL_RES`), so they open at the size of the raw image and overlay it
      directly in ImageJ or napari. Set that flag to False for the compact form instead, one
      TIFF pixel per patch.

    Row and column are the patch grid as drawn: row 0 is the top of the frame, and patch (i, j)
    covers image pixels [i*PATCH_SIZE : (i+1)*PATCH_SIZE, j*PATCH_SIZE : (j+1)*PATCH_SIZE]. NaN
    means no measurement there - either the patch never touched the mask, or its FRC produced no
    crossing - and `status` says which.
    """
    stem = image_stem(df)
    ny, nx = next(iter(maps.values())).shape
    written = []

    payload = dict(maps)
    payload.update(text_maps)
    payload.update({
        # Enough geometry to put a grid cell back on the image without recomputing anything.
        "patch_y0": np.arange(ny) * PATCH_SIZE,
        "patch_x0": np.arange(nx) * PATCH_SIZE,
        "extent_left_right_bottom_top": np.array([0, nx * PATCH_SIZE, ny * PATCH_SIZE, 0]),
        "patch_size_px": np.array(PATCH_SIZE),
        "px_nm": np.array(float(df["px_nm"].iloc[0])),
        "image_shape": np.array(image_shape[:2]),
        "source_image": np.array(df["path"].iloc[0]),
        "quantities": np.array(list(maps) + list(text_maps)),
    })
    npz_path = os.path.join(HEATMAP_DATA_DIR, stem + ".npz")
    np.savez_compressed(npz_path, **payload)
    written.append(npz_path)

    if HEATMAP_DATA_CSV:
        for k, m in list(maps.items()) + list(text_maps.items()):
            grid = pd.DataFrame(m, index=pd.RangeIndex(ny, name="row"),
                                columns=[f"col_{j}" for j in range(nx)])
            p = os.path.join(HEATMAP_DATA_DIR, f"{stem}__{k}.csv")
            grid.to_csv(p)
            written.append(p)

    if HEATMAP_DATA_TIF:
        # Written as an ImageJ TIFF with the pixel size in its tags, so a full-size map lands on
        # the image it came from at the right scale. The upsampled grids are blocky, so zlib
        # takes them back to about the size of the patch-sized version.
        px_um = float(df["px_nm"].iloc[0]) * (1 if HEATMAP_TIF_FULL_RES else PATCH_SIZE) / 1000.0
        res = (1.0 / px_um, 1.0 / px_um) if px_um > 0 else None
        for k, m in maps.items():
            p = os.path.join(HEATMAP_DATA_DIR, f"{stem}__{k}.tif")
            arr = grid_to_image_raster(m, image_shape) if HEATMAP_TIF_FULL_RES \
                else np.asarray(m, dtype=np.float32)
            tifffile.imwrite(p, arr, imagej=True, resolution=res, metadata={"unit": "um"},
                             compression="zlib")
            written.append(p)

    return written


def heatmap_figure(df, image, mask, maps):
    """The four-panel heatmap for one image -> (figure, filename)."""
    ny, nx = maps["r_co1_nm"].shape
    # The patch grid covers whole patches only, so it stops short of the right and bottom edges
    # of the frame. Drawing it in image pixel coordinates and then fixing the axes to the frame
    # keeps every panel registered to the image and to the mask outline.
    extent = [0, nx * PATCH_SIZE, ny * PATCH_SIZE, 0]

    # Drawn from a decimated copy: these panels are a visual guide a few hundred pixels across,
    # and rendering the full frame plus a full-resolution contour for every image costs minutes
    # over a folder. HEATMAP_DRAW_STEP affects nothing that was measured.
    DRAW_STEP = max(1, int(HEATMAP_DRAW_STEP))
    ext_img = [0, image.shape[1], image.shape[0], 0]
    small_mask = mask[::DRAW_STEP, ::DRAW_STEP].astype(float)

    fig, axes = plt.subplots(2, 2, figsize=(15, 9.5))
    lo, hi = np.percentile(image, [1, 99])
    axes[0, 0].imshow(image[::DRAW_STEP, ::DRAW_STEP], cmap="gray", vmin=lo, vmax=hi,
                      extent=ext_img)
    axes[0, 0].contour(small_mask, levels=[0.5], colors="yellow", linewidths=1.1,
                       extent=ext_img, origin="upper")
    for _, r in df.iterrows():
        axes[0, 0].add_patch(Rectangle((r["x0"], r["y0"]), PATCH_SIZE, PATCH_SIZE, fill=False,
                                       edgecolor="deepskyblue", linewidth=0.7))
    axes[0, 0].set_title(f"{len(df)} of {ny * nx} patches touch the mask", fontsize=10)

    panels = [(axes[0, 1], "mask_coverage", "fraction of the patch inside the cell", "cividis"),
              (axes[1, 0], "r_co1_nm", "$r_{co1}$ (nm) - raw checkerboard", "viridis_r"),
              (axes[1, 1], "r_cal_nm", "$r_{cal}$ (nm) - calibrated", "viridis_r")]
    for ax, key, ttl, cmap_name in panels:
        m = maps[key]
        cmap = plt.get_cmap(cmap_name).copy()
        cmap.set_bad("0.85")            # patches with no number, so the holes are legible
        finite = m[np.isfinite(m)]
        if key == "mask_coverage":
            vmin, vmax = 0.0, 1.0       # fixed, so coverage reads the same on every image
        elif finite.size:
            # Robust limits. A single badly behaved boundary patch would otherwise take the whole
            # colour scale and flatten the pattern these maps exist to show. Outliers are clipped
            # in the drawing only - nothing is dropped from the data.
            vmin, vmax = np.percentile(finite, [2, 98])
        else:
            vmin = vmax = None
        im = ax.imshow(np.ma.masked_invalid(m), cmap=cmap, extent=extent, interpolation="nearest",
                       vmin=vmin, vmax=vmax)
        ax.contour(small_mask, levels=[0.5], colors="crimson", linewidths=0.9,
                   extent=ext_img, origin="upper")
        plt.colorbar(im, ax=ax, fraction=0.035, pad=0.02)
        n_ok = int(np.isfinite(m).sum())
        clipped = "" if key == "mask_coverage" else "  (colour clipped to 2-98%)"
        ax.set_title(f"{ttl}   [{n_ok}/{len(df)} measured]{clipped}", fontsize=9.5)

    for ax in axes.ravel():
        ax.set_xlim(0, image.shape[1])
        ax.set_ylim(image.shape[0], 0)
        ax.set_aspect("equal")
        ax.set_xticks([])
        ax.set_yticks([])

    rel = os.path.join(df["rel_dir"].iloc[0], df["file"].iloc[0])
    px = df["px_nm"].iloc[0]
    fig.suptitle(f"{rel}   -   {px:.1f} nm/px, {PATCH_SIZE} px patches", fontsize=12)
    fig.tight_layout(rect=[0, 0, 1, 0.97])
    return fig, image_stem(df) + ".png"


SAVE_DATA = SAVE_HEATMAP_DATA and bool(HEATMAP_DATA_DIR)

if patch_df.empty:
    print("No patches measured - nothing to draw or save.")
elif not (SAVE_HEATMAPS or SAVE_DATA):
    print("SAVE_HEATMAPS and SAVE_HEATMAP_DATA are both off - Step 6 skipped.")
else:
    if SAVE_HEATMAPS and HEATMAP_DIR:
        os.makedirs(HEATMAP_DIR, exist_ok=True)
    if SAVE_DATA:
        os.makedirs(HEATMAP_DATA_DIR, exist_ok=True)

    t0 = time.time()
    written, data_files = [], []
    for n, (path, g) in enumerate(patch_df.groupby("path", sort=True), 1):
        # The grids and the panels are built from the same `maps`, so what is saved is exactly
        # what is drawn - the figure cannot drift away from the file beside it.
        image = load_image_2d(path)
        maps = patch_maps(g, image.shape)

        if SAVE_DATA:
            data_files += save_patch_grids(g, maps, patch_text_maps(g, image.shape), image.shape)

        if SAVE_HEATMAPS:
            fig, name = heatmap_figure(g, image, load_cell_mask(find_cell_mask(path)), maps)
            if HEATMAP_DIR:
                fig.savefig(os.path.join(HEATMAP_DIR, name), dpi=140)
                written.append(name)
            if n > HEATMAPS_SHOWN_INLINE:
                plt.close(fig)

    if SAVE_HEATMAPS:
        print(f"{len(written)} heatmap(s) written to "
              f"{HEATMAP_DIR or '(nowhere - OUTPUT_PATH unset)'} in {time.time() - t0:.0f} s")
        for name in written[:4]:
            print("   ", name)
        if len(written) > 4:
            print(f"    ... and {len(written) - 4} more")

    if SAVE_DATA:
        n_npz = sum(f.endswith(".npz") for f in data_files)
        print(f"\n{len(data_files)} grid file(s) for {n_npz} image(s) written to {HEATMAP_DATA_DIR}")
        print(f"  quantities: {', '.join(GRID_KEYS + GRID_TEXT_KEYS)}")
        if HEATMAP_DATA_TIF:
            print("  TIFFs are %s" % ("full size - same pixel grid as the raw image, so they "
                                     "overlay it directly" if HEATMAP_TIF_FULL_RES else
                                     "one pixel per patch (HEATMAP_TIF_FULL_RES is off)"))
        print("  These are the values themselves - unclipped, NaN where a patch has no number. "
              "The PNG\n  colour scale is clipped to 2-98% and cannot be read back, which is what "
              "these are for.")
        print("\n  Read one back with:")
        print(f'      d = np.load(r"{data_files[0]}")')
        print('      plt.imshow(d["r_cal_nm"], interpolation="nearest")')
        if HEATMAP_DATA_CSV:
            print('      # or, from the CSV:  pd.read_csv(path, index_col=0).to_numpy()')

    if SAVE_HEATMAPS and written:
        print(f"\nThe first {min(HEATMAPS_SHOWN_INLINE, len(written))} are drawn below; the rest "
              f"are on disk only. Raise HEATMAPS_SHOWN_INLINE to see more inline.")
        plt.show()


## Step 7 - summary and CSV output

Three tables. `..._patches_256.csv` is the primary record - one row per patch touching the cell
mask, at the scale the patch calibration was fitted for, with its own `mask_coverage`,
`status` and `calib_range`. `..._whole_region.csv` is the same measurement over the whole masked
region, the direct analogue of the previous notebook's in-mask number. `..._summary.csv`
collapses both to one row per image.

The summary carries **n, mean, median, std, min, max and IQR** for three groups of patches per
image: the raw `r_co1`, the calibrated `r_cal`, and `r_cal` restricted to the patches whose
`r_co1` fell inside the fitted range. Reporting all of them is deliberate. The median and IQR are
the pair to trust here, because the grid includes boundary patches that read far from the
interior ones and a mean and standard deviation are both sensitive to exactly that; but a mean
and std are what a normal-error analysis expects, and min/max show how far the tails reach, so
none of them is withheld. Only a subset is printed on screen - the CSV has the lot.

If the all-patch and in-range medians differ materially, the difference is the calibration
extrapolating rather than the specimen varying, and the in-range one is the honest number.

A **coverage check** is printed at the end, and it is the first thing to read. A calibration is
an interpolation between the points it was fitted on; asked for a value beyond them it does not
fail, it saturates - the exponential flattens onto `d` and returns very nearly the same number
whatever it is given. The symptom is unmistakable once looked for: calibrated values that agree
to a fraction of a nanometre across images whose raw `r_co1` still varies by several. If that is
what the check reports, the correct conclusion is that this dataset lies outside the calibration,
not that every field happens to share a resolution.

In [ ]:
IN_RANGE = "in_range"

# Every statistic is reported rather than one being chosen here. Median and IQR are the robust
# pair, and they are what the text below quotes: a single boundary patch reading 10 nm fine drags
# a mean and inflates a standard deviation, and this patch grid has plenty of those. But mean and
# std are what a normal-error analysis expects, and min/max say how far the extremes reach, so
# all of them go into the CSV and the choice of which to quote stays with whoever reads it.
STAT_FNS = {
    "n":      lambda s: float(s.size),
    "mean":   lambda s: float(s.mean()),
    "median": lambda s: float(s.median()),
    "std":    lambda s: float(s.std(ddof=1)) if s.size > 1 else np.nan,
    "min":    lambda s: float(s.min()),
    "max":    lambda s: float(s.max()),
    "iqr":    lambda s: float(s.quantile(0.75) - s.quantile(0.25)),
}


def stats_block(values, prefix):
    """STAT_FNS of *values* with NaNs dropped -> {'<prefix>_<stat>': value}."""
    s = pd.Series(values, dtype=float).dropna()
    if s.empty:
        return {f"{prefix}_{k}": (0.0 if k == "n" else np.nan) for k in STAT_FNS}
    return {f"{prefix}_{k}": f(s) for k, f in STAT_FNS.items()}


def _med(s):
    s = s.dropna()
    return float(s.median()) if len(s) else np.nan


summary_rows = []
for path, grp in (patch_df.groupby("path") if not patch_df.empty else []):
    ok = grp[grp["status"] == "ok"]
    inr = ok[ok["calib_range"] == IN_RANGE]
    w = whole_df[whole_df["path"] == path]
    w = w.iloc[0] if len(w) else {}
    summary_rows.append({
        "file": grp["file"].iloc[0], "rel_dir": grp["rel_dir"].iloc[0], "path": path,
        "px_nm": grp["px_nm"].iloc[0],
        "n_patches": len(grp), "n_patches_measured": len(ok), "n_patches_in_range": len(inr),
        "patch_mask_coverage_median": _med(grp["mask_coverage"]),
        # Three blocks: the raw measurement, the calibrated one, and the calibrated one restricted
        # to patches the calibration actually covers. The third is the honest one wherever the
        # coverage check in this cell reports widespread extrapolation.
        **stats_block(ok["r_co1_nm"], "patch_r_co1_nm"),
        **stats_block(ok["r_cal_nm"], "patch_r_cal_nm"),
        **stats_block(inr["r_cal_nm"], "patch_r_cal_nm_in_range"),
        "region_r_co1_nm": w.get("r_co1_nm", np.nan),
        "region_r_cal_nm": w.get("r_cal_nm", np.nan),
        "region_status": w.get("status", ""), "region_calib_range": w.get("calib_range", ""),
        "region_mask_coverage": w.get("mask_coverage", np.nan),
    })

summary_df = pd.DataFrame(summary_rows)

# The full table is too wide to read on screen, so a subset is displayed and everything is written
# to the CSV. Nothing is dropped, only hidden here.
DISPLAY_COLS = (["rel_dir", "file", "n_patches", "n_patches_measured", "n_patches_in_range"]
                + [f"patch_{q}_nm_{s}" for q in ("r_co1", "r_cal")
                   for s in ("mean", "median", "std", "min", "max")]
                + ["region_r_co1_nm", "region_r_cal_nm"])
print("=== PER-IMAGE SUMMARY ===")
print("(the CSV also carries IQR, the in-range block, coverage and the region flags)")
display(summary_df[[c for c in DISPLAY_COLS if c in summary_df.columns]])

if len(summary_df):
    ok_patches = patch_df[patch_df["status"] == "ok"]
    inr_patches = ok_patches[ok_patches["calib_range"] == IN_RANGE]
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))

    axes[0].hist(ok_patches["r_cal_nm"].dropna(), bins=40, color="steelblue", alpha=0.55,
                 label=f"all calibrated patches (n = {ok_patches['r_cal_nm'].notna().sum()})")
    axes[0].hist(inr_patches["r_cal_nm"].dropna(), bins=40, color="darkorange", alpha=0.7,
                 label=f"inside the fitted range (n = {inr_patches['r_cal_nm'].notna().sum()})")
    for c, cal in CALIB.items():
        if c == PATCH_SCALE and np.isfinite(cal["ceiling_px"]):
            axes[0].axvline(cal["ceiling_px"] * summary_df["px_nm"].iloc[0], color="crimson",
                            ls="--", lw=1.2, label="calibration ceiling")
    axes[0].set_xlabel("$r_{cal}$ (nm)")
    axes[0].set_ylabel("patches")
    axes[0].set_title(f"Calibrated resolution, {PATCH_SIZE} px patches inside the cell", fontsize=10)
    axes[0].legend(fontsize=8)
    axes[0].grid(alpha=0.25)

    # Raw and calibrated on the same axis, per image. Bars would hide the point when the
    # calibration saturates - every image would draw the same length - so both are plotted as
    # points and the gap between them is the correction that was applied.
    s = summary_df.sort_values("patch_r_co1_nm_median")
    y = np.arange(len(s))
    axes[1].errorbar(s["patch_r_cal_nm_median"], y, xerr=s["patch_r_cal_nm_iqr"] / 2, fmt="o",
                     ms=5, color="steelblue", lw=0.8, capsize=2,
                     label="patch $r_{cal}$ median (bar = IQR/2)")
    axes[1].scatter(s["patch_r_co1_nm_median"], y, s=26, color="grey", alpha=0.8, zorder=3,
                    label="patch $r_{co1}$ median (uncalibrated)")
    axes[1].scatter(s["region_r_cal_nm"], y, s=26, marker="s", color="darkorange", zorder=3,
                    label="whole region $r_{cal}$")
    for scale, cal in CALIB.items():
        if scale == PATCH_SCALE:
            axes[1].axvspan(cal["x_min"] * s["px_nm"].iloc[0], cal["x_max"] * s["px_nm"].iloc[0],
                            color="olive", alpha=0.10, label="calibrated range of $r_{co1}$")
    axes[1].set_yticks(y)
    axes[1].set_yticklabels([os.path.join(r, f) for r, f in zip(s["rel_dir"], s["file"])],
                            fontsize=6)
    axes[1].set_xlabel("resolution (nm)")
    axes[1].set_xlim(left=0)
    axes[1].set_title("Per image, sorted by raw measurement: what the calibration did", fontsize=10)
    axes[1].legend(fontsize=7)
    axes[1].grid(alpha=0.25, axis="x")

    fig.tight_layout()
    if OUTPUT_PATH and SAVE_FIGURES:
        os.makedirs(os.path.join(OUTPUT_PATH, "figures"), exist_ok=True)
        fig.savefig(os.path.join(OUTPUT_PATH, "figures", "calibrated_resolution.png"), dpi=150)
    plt.show()

    all_ok = ok_patches["r_cal_nm"].dropna()
    in_ok = inr_patches["r_cal_nm"].dropna()
    print(f"Across every measured patch: r_cal median {all_ok.median():.2f} nm "
          f"(IQR {all_ok.quantile(0.25):.2f} - {all_ok.quantile(0.75):.2f})")
    if len(in_ok):
        print(f"Restricted to the fitted range: {in_ok.median():.2f} nm "
              f"(IQR {in_ok.quantile(0.25):.2f} - {in_ok.quantile(0.75):.2f}), n = {len(in_ok)}")
    print(f"Raw, uncalibrated for comparison : {ok_patches['r_co1_nm'].median():.2f} nm "
          f"-> the calibration moved the median by "
          f"{all_ok.median() - ok_patches['r_co1_nm'].median():+.2f} nm")

    print("\nEvery statistic, over all measured patches (nm):")
    print("  quantity              n    mean  median     std     min     max     IQR")
    for lbl, s in (("r_co1 (raw)", ok_patches["r_co1_nm"]),
                   ("r_cal (calibrated)", ok_patches["r_cal_nm"]),
                   ("r_cal (in range)", inr_patches["r_cal_nm"])):
        b = stats_block(s, "x")
        print("  %-18s %5.0f  %6.2f  %6.2f  %6.2f  %6.2f  %6.2f  %6.2f"
              % (lbl, b["x_n"], b["x_mean"], b["x_median"], b["x_std"], b["x_min"], b["x_max"],
                 b["x_iqr"]))
    print("  A std on the r_cal row is not a measurement precision: it is mostly the width of the\n"
          "  pile-up against the calibration ceiling, so it shrinks as the saturation gets worse.")

    spread = summary_df["patch_r_cal_nm_iqr"].median()
    print(f"\nPatch-to-patch IQR within an image, median {spread:.2f} nm. That, not the fit "
          f"residual,\nsets the smallest resolution difference this method can honestly resolve.")

    # Does the calibration actually cover this dataset? Everything above rests on that, so it is
    # checked and stated rather than left for the reader to infer from a flag column.
    cal = CALIB.get(PATCH_SCALE)
    if cal is not None and len(ok_patches):
        px = float(summary_df["px_nm"].median())
        frac_hi = float((ok_patches["calib_range"] == "extrapolated_high").mean())
        print(f"\n{'-' * 78}\nCoverage check: {100 * frac_hi:.0f}% of the measured patches are "
              f"coarser than the calibrated range.")
        print(f"At {px:.1f} nm/px the {PATCH_SIZE} px calibration covers r_co1 = "
              f"{cal['x_min'] * px:.1f} - {cal['x_max'] * px:.1f} nm; this dataset measures a "
              f"median of {ok_patches['r_co1_nm'].median():.1f} nm.")
        if frac_hi > 0.5:
            print("\nThat is a finding rather than a detail. These images are coarser than "
                  "anything the\ncalibration was fitted on, the exponential has flattened onto "
                  f"its ceiling of\n{cal['ceiling_px'] * px:.1f} nm, and r_cal there is that "
                  "ceiling rather than a measurement - which is\nwhy the calibrated values above "
                  "are all within a fraction of a nanometre of each other\nwhile the raw r_co1 "
                  "values still vary. Two defensible ways forward: quote r_co1 and say it\nis "
                  "uncalibrated in this regime, or extend the calibration with repeat pairs at "
                  "this\nresolution so the fit spans it. Do not read the ceiling as a resolution.")
        else:
            print("Most points fall inside the fitted range, so r_cal is interpolation and the "
                  "medians above stand.")
        print("-" * 78)

if OUTPUT_PATH:
    os.makedirs(OUTPUT_PATH, exist_ok=True)
    calib_used = pd.DataFrame([{"scale": k, "variant": CALIBRATION_VARIANT,
                                "a": v["params"][0], "b": v["params"][1],
                                "c": v["params"][2], "d": v["params"][3],
                                "x_min": v["x_min"], "x_max": v["x_max"],
                                "ceiling_px": v["ceiling_px"], "resid_std": v["resid_std"],
                                "r2": v["r2"], "n_fit": v["n"], "source": fits_path}
                               for k, v in CALIB.items()])
    written = [
        (f"frc_checkerboard_in_mask_patches_{PATCH_SIZE}.csv", patch_df),
        ("frc_checkerboard_in_mask_whole_region.csv", whole_df),
        ("frc_checkerboard_in_mask_summary.csv", summary_df),
        ("calibration_used.csv", calib_used),
    ]
    print(f"\nWriting to {OUTPUT_PATH}")
    for name, df in written:
        df.to_csv(os.path.join(OUTPUT_PATH, name), index=False)
        print(f"  {name:52s} {len(df):6d} rows x {df.shape[1]:2d} columns")
    print("\n`r_cal_nm` is the number to quote. `r_co1_nm` is the raw checkerboard measurement it "
          "came\nfrom, `calib_range` says whether the correction was interpolated, and "
          "`ceiling_frac` says how\nclose the result sits to the coarsest value the calibration "
          "can express.")
else:
    print("\nOUTPUT_PATH is empty - nothing written.")